# 02 Hypotheses and Evaluation

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/vankadaruvijayberk/HypotheSAEs.git'   # <-- your fork
REPO_DIR = '/content/HypotheSAEs'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Install

In [2]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

!pip install -q -e {REPO_DIR}
# Added "numpy<2.0.0" to fix the scipy ufunc binary compatibility error
!pip install -q -U datasets sentence-transformers transformers scikit-learn statsmodels "numpy<2.0.0"

# Install the latest vLLM to support Qwen3, plus the CUDA runtime to prevent the libcudart error!
# We also upgrade torchaudio/torchvision so they stay in sync with the new PyTorch version.
!pip install -q -U vllm torchaudio torchvision nvidia-cuda-runtime-cu12


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hypothesaes (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.25.0 requires torchcodec>=0.14, which is not installed.
outlines 0.1.11 requires outlines_core==0.1.26, but you have outlines-core 0.2.14 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xformers 0.0.28.post3 requires torch==2.5.1, but you have torch 2.11.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, bu

In [3]:
# editable-install .pth files are only read at interpreter startup, so add the paths directly
import os, sys, importlib

for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

In [4]:
# Uninstall torchcodec to resolve the libnvrtc.so / PyTorch compatibility conflict
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.14.0
Uninstalling torchcodec-0.14.0:
  Successfully uninstalled torchcodec-0.14.0


## Imports

In [5]:
import numpy as np
import pandas as pd
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.quickstart import train_sae
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import (NeuronInterpreter, InterpretConfig,
                                           SamplingConfig, LLMConfig, ScoringConfig)
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses
ge.set_seed()
ge.redirect_annotation_cache()

## Start Server

In [6]:
# The downgrade and dependency fixes have been moved to the main Install cell at the top!

In [7]:
import subprocess, time, requests, os

MODEL = 'Qwen/Qwen3-1.7B'
PORT = 8000

# Open a file to capture the logs so Colab doesn't hide them
log_file = open('vllm_logs.txt', 'w')

_server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--port', str(PORT),
     '--max-model-len', '8192', '--gpu-memory-utilization', '0.85'],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

BASE = f'http://127.0.0.1:{PORT}/v1'

for _ in range(180):
    # Check if the server process crashed prematurely
    if _server.poll() is not None:
        log_file.close()
        # Read and display the logs!
        with open('vllm_logs.txt', 'r') as f:
            logs = f.read()
        raise RuntimeError(f"vLLM server crashed! Here are the logs:\n{'='*40}\n{logs}\n{'='*40}")

    try:
        if requests.get(BASE + '/models', timeout=2).status_code == 200:
            print('\nServer up at', BASE)
            break
    except Exception:
        pass
    time.sleep(5)
else:
    log_file.close()
    raise RuntimeError('vLLM server did not start within the time limit.')

os.environ['OPENAI_BASE_URL'] = BASE
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')


Server up at http://127.0.0.1:8000/v1


'EMPTY'

## Smoke Test

In [8]:
from hypothesaes.llm_api import get_completion
print(get_completion(prompt='Reply with exactly: hello', model=MODEL,
                     max_output_tokens=2048, temperature=0.0))

<think>
Okay, the user wants me to reply with exactly "hello". Let me make sure I don't add anything else. Just a simple "hello" without any extra text. I need to keep it concise. Maybe check if there's any hidden instruction, but the prompt says to reply with exactly that. Alright, done.
</think>

hello


## Load Data, Embeddings, SAE

In [9]:
ds, NAMES = ge.load_goemotions()
train_texts = list(ds['train']['text'])
test_texts  = list(ds['test']['text'])
train_targets = ge.build_targets(ds['train']['labels'], NAMES)
test_targets  = ge.build_targets(ds['test']['labels'], NAMES)

t2e = get_local_embeddings(train_texts, model=ge.EMBEDDER, nomic_prefix=ge.NOMIC_PREFIX,
                           cache_name='goemotions_modernbert', batch_size=128)
train_emb = np.stack([t2e[t] for t in train_texts]).astype(np.float32)
del t2e; ge.clear_mem()

sae = train_sae(embeddings=train_emb, M=1024, K=8,
                checkpoint_dir=os.path.join(ge.CKPT_DIR, 'goemotions_M1024_K8'))
acts = sae.get_activations(train_emb)
del train_emb; ge.clear_mem()
acts.shape

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Loading embedding chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded 53994 embeddings in 9.7s
Loaded model from /content/drive/MyDrive/hypothesaes_goemotions/checkpoints/goemotions_M1024_K8/SAE_M=1024_K=8.pt onto device cuda


Computing activations (batchsize=16384):   0%|          | 0/3 [00:00<?, ?it/s]

(43410, 1024)

## Select Neurons

In [10]:
N_SELECT = 10
selected, sel_scores = {}, {}
for t in ge.TARGETS:
    if train_targets[t].sum() < 20:
        continue
    idxs, scores = select_neurons(activations=acts, target=train_targets[t],
                                  n_select=N_SELECT, method='correlation', classification=True)
    selected[t] = [int(i) for i in idxs]
    sel_scores[t] = [float(s) for s in scores]
union = sorted({i for v in selected.values() for i in v})
ge.log_json('05_selection', {'n_select': N_SELECT, 'selected': selected,
                             'scores': sel_scores, 'union_size': len(union)})
print('targets:', list(selected), '| union:', len(union))

targets: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'curiosity', 'disappointment'] | union: 65


## Interpret Neurons

In [11]:
interpreter = NeuronInterpreter(interpreter_model=MODEL, annotator_model=MODEL,
                                n_workers_interpretation=8, n_workers_annotation=16,
                                cache_name='goemotions')
icfg = InterpretConfig(
    sampling=SamplingConfig(n_examples=20, max_words_per_example=64),
    llm=LLMConfig(temperature=0.7, max_output_tokens=4096),
    n_candidates=3,
    task_specific_instructions=ge.TASK_INSTRUCTIONS,
)
interps = interpreter.interpret_neurons(texts=train_texts, activations=acts,
                                        neuron_indices=union, config=icfg)

Generating interpretations:   0%|          | 0/195 [00:00<?, ?it/s]

## Score Fidelity

In [12]:
scfg = ScoringConfig(n_examples=100, max_words_per_example=64)
metrics = interpreter.score_interpretations(texts=train_texts, activations=acts,
                                            interpretations=interps, config=scfg, **ge.NO_THINK)

Found 0 cached items; annotating 16700 uncached items


Scoring neuron interpretation fidelity (65 neurons; 3 candidate interps per neuron; 100 examples to score each…

## Best Interpretations

In [13]:
best = {}
for n in union:
    cands = [c for c in interps[n] if c]
    if not cands:
        best[n] = {'interpretation': None, 'f1': 0.0}
        continue
    bi = max(cands, key=lambda c: metrics[n][c]['f1'])
    best[n] = {'interpretation': bi, 'f1': float(metrics[n][bi]['f1'])}

neuron2hyp = {n: v['interpretation'] for n, v in best.items() if v['interpretation']}
ge.log_json('06_interpretations', {'best': {str(k): v for k, v in best.items()},
                                   'selected': selected})
del acts; ge.clear_mem()
pd.DataFrame([{'neuron': n, **v} for n, v in best.items()]).head(10)

,neuron,interpretation,f1
0,4,mentions the word 'boring' in the text,0.813953
1,9,mentions of specific sports teams,0.113208
2,36,"contains a direct command (e.g., 'shut up', 's...",0.901961
3,41,mentions thank you,0.470588
4,61,mentions 'beautiful',0.948454
5,66,absence of action,0.730159
6,78,asks a direct question seeking information,0.947368
7,94,contains the word 'creepy',0.804598
8,114,mentions funny,0.724138
9,117,mentions a specific person,0.075472


## Annotate Heldout

In [14]:
rng = np.random.default_rng(ge.SEED)
hidx = rng.choice(len(test_texts), size=min(ge.HELDOUT_CAP, len(test_texts)), replace=False)
held_texts = [test_texts[i] for i in hidx]
held_targets = {t: test_targets[t][hidx] for t in test_targets}

union_hyps = sorted(set(neuron2hyp.values()))
ann = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model=MODEL,
                                   cache_name='goemotions_heldout', n_workers=16,
                                   max_words_per_example=64, **ge.NO_THINK)

Found 0 cached items; annotating 128000 uncached items


Annotating:   0%|          | 0/128000 [00:00<?, ?it/s]

## Predictiveness

In [15]:
results, rows = {}, []
for t, neurons in selected.items():
    hyps = sorted({neuron2hyp[n] for n in neurons if n in neuron2hyp})
    if len(hyps) < 2:
        continue
    m, df = score_hypotheses({h: ann[h] for h in hyps}, y_true=held_targets[t].astype(float),
                             classification=True, corrected_pval_threshold=0.1)
    results[t] = {'auroc': m.get('auroc'), 'r2': m.get('r2'),
                  'significant': m['Significant'][0], 'n_hyp': m['Significant'][1]}
    df['target'] = t
    rows.append(df)
table = pd.concat(rows, ignore_index=True)
ge.log_json('07_evaluation', {'per_target': results,
                              'heldout_indices': [int(i) for i in hidx],
                              'hypotheses': table.to_dict('records')})
pd.DataFrame(results).T

Optimization terminated successfully.
         Current function value: 0.346368
         Iterations 8
         Current function value: 0.081046
         Iterations: 35
         Current function value: inf
         Iterations: 35
Error fitting model: Singular matrix, trying OLS instead
Optimization terminated successfully.
         Current function value: 0.588542
         Iterations 6
         Current function value: 0.234536
         Iterations: 35
         Current function value: 0.310284
         Iterations: 35
Optimization terminated successfully.
         Current function value: 0.146223
         Iterations 8


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to

Optimization terminated successfully.
         Current function value: 0.118376
         Iterations 8


,auroc,r2,significant,n_hyp
anger,0.709829,0.096884,3.0,9.0
disgust,0.794579,0.173329,1.0,10.0
fear,0.817462,NaN,4.0,10.0
joy,0.724161,0.120810,6.0,10.0
sadness,0.715175,0.115381,3.0,10.0
surprise,0.738085,0.154368,2.0,10.0
curiosity,0.785690,0.252288,2.0,10.0
disappointment,0.738079,0.085816,1.0,10.0


## SAE vs Interpreter

In [16]:
sel_score = {(t, n): s for t in selected for n, s in zip(selected[t], sel_scores[t])}
recs = []
for t, neurons in selected.items():
    for n in neurons:
        h = neuron2hyp.get(n)
        if not h:
            continue
        sep = table[(table.target == t) & (table.hypothesis == h)]['separation_score']
        recs.append({'target': t, 'neuron': n, 'hypothesis': h,
                     'train_corr': abs(sel_score[(t, n)]),
                     'fidelity_f1': best[n]['f1'],
                     'heldout_sep': float(sep.iloc[0]) if len(sep) else None})
d8 = pd.DataFrame(recs)

def diagnose(r):
    if r['fidelity_f1'] < 0.5 and r['train_corr'] > 0.05:
        return 'interpreter_bottleneck'
    if r['train_corr'] < 0.03:
        return 'weak_feature'
    return 'ok'

d8['diagnosis'] = d8.apply(diagnose, axis=1)
ge.log_json('08_failure_localization', {'records': d8.to_dict('records'),
                                        'counts': d8['diagnosis'].value_counts().to_dict()})
print(d8['diagnosis'].value_counts().to_dict())
d8.sort_values('fidelity_f1').head(10)

{'ok': 66, 'interpreter_bottleneck': 13}


,target,neuron,hypothesis,train_corr,fidelity_f1,heldout_sep,diagnosis
62,curiosity,500,mentions specific topics related to research o...,0.124456,0.000000,0.451451,interpreter_bottleneck
52,surprise,1014,mentions 'argonians',0.136136,0.000000,-0.120060,interpreter_bottleneck
58,surprise,500,mentions specific topics related to research o...,0.075783,0.000000,0.380380,interpreter_bottleneck
43,sadness,529,mentions a specific pet (cat),0.108428,0.039216,-0.074649,interpreter_bottleneck
9,disgust,117,mentions a specific person,0.338507,0.075472,-0.020747,interpreter_bottleneck
21,fear,117,mentions a specific person,0.143152,0.075472,-0.001153,interpreter_bottleneck
73,disappointment,117,mentions a specific person,0.074756,0.075472,-0.000749,interpreter_bottleneck
28,fear,9,mentions of specific sports teams,0.047372,0.113208,-0.015068,ok
31,joy,132,mentions 'man' in a repeated way,0.173057,0.142857,-0.035094,interpreter_bottleneck
16,disgust,680,mentions NSFW,0.046974,0.241379,-0.020090,ok


## Worked Example

In [17]:
d8[d8.diagnosis == 'interpreter_bottleneck'].sort_values('train_corr', ascending=False).head(3)

,target,neuron,hypothesis,train_corr,fidelity_f1,heldout_sep,diagnosis
9,disgust,117,mentions a specific person,0.338507,0.075472,-0.020747,interpreter_bottleneck
31,joy,132,mentions 'man' in a repeated way,0.173057,0.142857,-0.035094,interpreter_bottleneck
21,fear,117,mentions a specific person,0.143152,0.075472,-0.001153,interpreter_bottleneck


## Shutdown

In [18]:
_server.terminate(); ge.clear_mem()